# 07 · Quantization from scratch: int8, int4 and NF4

After this notebook you can compute how much memory any model needs at any precision, implement symmetric and zero-point quantization, explain why granularity (per-tensor, per-channel, block-wise) and outliers matter so much, build the NF4 data type of QLoRA from normal quantiles (and match bitsandbytes bit for bit), and read the GPTQ / AWQ / GGUF / bitsandbytes landscape.

**Time:** ~35 min · **Runs:** offline on CPU in < 1 min · **Needs:** [NumPy for AI engineers](../04_numpy_pandas_pytorch/01_numpy_for_ai_engineers.ipynb), [04 · Tiny GPT](04_transformer_block_and_tiny_gpt.ipynb) (parameter counts)

## Why this matters in interviews

- "Will a 70B model fit on one 80 GB GPU?" is arithmetic you must do on the spot: parameters x bytes, plus the KV cache.
- Quantization is the biggest single cost lever for self-hosting; you need to know what it costs in quality and which method to choose.
- QLoRA questions (NF4, double quantization, why 4-bit base + 16-bit adapters) are standard for fine-tuning roles.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `fo30` | What is quantisation, in one paragraph, and what does it cost you? |
| `in07` | What is quantisation and which method would you pick? |
| `tr29` | How do you compute the memory a model needs to serve? |
| `tr42` | The notebook says the memory footprint of this 3B model is 12.9 GB. Why, and how would you cut it? |
| `in30` | How would you serve a 70B model on a single 80 GB GPU? |
| `tr33` | What is the difference between weight quantisation and KV cache quantisation? |
| `tn05` | LoRA vs QLoRA — what exactly is the difference? |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from scipy.stats import norm

rng = np.random.default_rng(0)
torch.manual_seed(0)
torch.set_num_threads(1)          # tiny model: one thread is fastest and immune to a busy CPU
pd.set_option("display.width", 120)
print("ready")

## 1. Why quantize: memory = parameters x bytes per parameter

**In plain English:** a model is billions of numbers. Stored as 32-bit floats each takes 4 bytes; as 16-bit, 2 bytes; as 8-bit integers, 1 byte; as 4-bit, half a byte. Fewer bytes means it fits on a smaller GPU, loads faster, and - because decoding is memory-bandwidth-bound (notebook 06) - generates tokens faster. The price is rounding error.

In [ ]:
BYTES = {"fp32": 4, "fp16/bf16": 2, "int8": 1, "int4": 0.5}
sizes = {"7B": 7e9, "13B": 13e9, "70B": 70e9}
mem = pd.DataFrame({dtype: {name: n * b / 1e9 for name, n in sizes.items()} for dtype, b in BYTES.items()})
print("weights only, GB (1 GB = 1e9 bytes):")
print(mem.round(1).to_string())
assert mem.loc["7B", "fp16/bf16"] == 14.0 and mem.loc["70B", "int4"] == 35.0

x = np.arange(len(sizes)); w = 0.2
fig, ax = plt.subplots(figsize=(9, 3.8))
for j, dtype in enumerate(BYTES):
    ax.bar(x + (j - 1.5) * w, mem[dtype], w, label=dtype)
for cap, name in [(24, "24 GB consumer GPU"), (80, "80 GB A100/H100")]:
    ax.axhline(cap, color="gray", ls="--", lw=1); ax.text(2.45, cap * 1.08, name, fontsize=8, ha="right")
ax.set_yscale("log"); ax.set_xticks(x, sizes); ax.set_ylabel("GB (log scale)")
ax.set_title("Weight memory by model size and precision"); ax.legend(fontsize=8)
plt.show()
print(f"tr42: a 3.21B model in fp32 = {3.21e9 * 4 / 1e9:.2f} GB (the '12.9 GB'), in bf16 {3.21e9 * 2 / 1e9:.1f} GB, "
      f"in 4-bit ~{3.21e9 * 0.5 / 1e9:.1f} GB")

A 70B model does not fit on one 80 GB GPU in fp16 (140 GB), fits in int8 with little room for the KV cache (70 GB), and fits comfortably in 4-bit (35 GB + cache). Serving memory is always **weights + KV cache + activations/overhead** (~10-20%).

## 2. Symmetric (absmax) int8 quantization

Map the range [-max|w|, +max|w|] onto the integers [-127, 127]:

`scale = max|w| / 127`,  `q = round(w / scale)`,  `w_hat = q x scale`

The integers are stored; the single float `scale` is stored alongside. Rounding error per value is at most scale/2.

In [ ]:
def quant_absmax(w, bits=8, axis=None):
    """Symmetric quantization. axis=None: one scale for the whole tensor; axis=1: one scale per row."""
    qmax = 2 ** (bits - 1) - 1
    amax = np.abs(w).max() if axis is None else np.abs(w).max(axis=axis, keepdims=True)
    scale = np.where(amax == 0, 1.0, amax / qmax)
    q = np.clip(np.round(w / scale), -qmax, qmax).astype(np.int8 if bits <= 8 else np.int16)
    return q, scale

def dequant(q, scale, zero=0):
    return (q.astype(np.float32) - zero) * scale

w = rng.normal(0, 0.5, size=8).astype(np.float32)
q, s = quant_absmax(w)
w_hat = dequant(q, s)
print(pd.DataFrame({"w": w, "q (int8)": q, "w_hat": w_hat, "error": w - w_hat}).round(4).to_string(index=False))
print(f"scale = {float(s):.5f}; max |error| = {np.abs(w - w_hat).max():.5f} <= scale/2 = {float(s) / 2:.5f}")
assert np.abs(w - w_hat).max() <= float(s) / 2 + 1e-7

## 3. Asymmetric (zero-point) quantization

Symmetric quantization wastes half the codes when the data is all positive (e.g. activations after ReLU/GELU): nothing ever lands on the negative integers. **Asymmetric** quantization maps [min, max] onto [0, 255] with a *zero-point*:

`scale = (max - min) / 255`,  `zero = round(-min / scale)`,  `q = clip(round(w / scale) + zero, 0, 255)`

In [ ]:
def quant_zeropoint(w, bits=8):
    qmax = 2 ** bits - 1
    lo, hi = float(w.min()), float(w.max())
    scale = (hi - lo) / qmax
    zero = int(round(-lo / scale))
    q = np.clip(np.round(w / scale) + zero, 0, qmax).astype(np.uint8)
    return q, scale, zero

acts = np.maximum(rng.normal(0.5, 1.0, size=10_000), 0).astype(np.float32)     # ReLU-like: all >= 0
q_s, s_s = quant_absmax(acts)
q_a, s_a, z_a = quant_zeropoint(acts)
mse_sym = np.mean((acts - dequant(q_s, s_s)) ** 2)
mse_asym = np.mean((acts - dequant(q_a, s_a, z_a)) ** 2)
print(f"symmetric : codes used {len(np.unique(q_s))}/255, MSE {mse_sym:.2e}")
print(f"zero-point: codes used {len(np.unique(q_a))}/256, MSE {mse_asym:.2e} (zero-point = {z_a})")
assert mse_asym < mse_sym / 3

Using the full code range gives ~4x lower error on this data (the step size halves, error variance scales with its square). Weights are roughly symmetric around 0, so weight quantization is usually symmetric; activations often use zero-points.

## 4. Granularity: per-tensor vs per-channel vs block-wise

One scale for a whole matrix is only as good as the matrix's largest value allows. Real weight matrices have rows (output channels) with very different magnitudes, so finer granularity - one scale **per row**, or per **block of 64 values** - reduces error, at the cost of storing more scales.

In [ ]:
def quant_blockwise(w, bits=4, block=64):
    flat = w.reshape(-1, block)
    q, s = quant_absmax(flat, bits=bits, axis=1)
    return dequant(q, s).reshape(w.shape)

row_scale = np.exp(rng.normal(0, 0.8, size=(256, 1)))                         # channels differ in magnitude
W = (rng.normal(size=(256, 256)) * row_scale).astype(np.float32)

def rel_err(a, b):
    return float(np.linalg.norm(a - b) / np.linalg.norm(a))

rows = []
for bits in [8, 4]:
    per_tensor = dequant(*quant_absmax(W, bits))
    per_channel = dequant(*quant_absmax(W, bits, axis=1))
    block = quant_blockwise(W, bits, block=64)
    rows.append((bits, rel_err(W, per_tensor), rel_err(W, per_channel), rel_err(W, block)))
gran = pd.DataFrame(rows, columns=["bits", "per-tensor", "per-channel (row)", "block-wise (64)"]).set_index("bits")
print("relative error ||W - W_hat|| / ||W||:"); print(gran.round(4).to_string())

fig, ax = plt.subplots(figsize=(8, 3.6))
gran.T.plot.bar(ax=ax, rot=0)
ax.set_yscale("log"); ax.set_ylabel("relative error (log)")
ax.set_title("Finer granularity -> smaller error (same number of bits)"); ax.legend(title="bits")
plt.show()
for b in [8, 4]:
    assert gran.loc[b, "per-tensor"] > gran.loc[b, "per-channel (row)"] > gran.loc[b, "block-wise (64)"]
print(f"overhead of block-wise scales: one fp16 scale per 64 weights = {16 / 64:.2f} extra bits per weight")

## 5. The outlier problem, and LLM.int8's fix

One large value ruins a shared scale: the step size is set by the outlier, so all the ordinary values collapse onto a handful of codes (many to exactly 0).

In [ ]:
vals = rng.normal(0, 1, size=4096).astype(np.float32)
with_outlier = vals.copy(); with_outlier[123] = 60.0                         # a single outlier
for name, v in [("no outlier", vals), ("one outlier (60)", with_outlier)]:
    q, s = quant_absmax(v)
    others = np.delete(np.arange(len(v)), 123)
    err = rel_err(v[others], dequant(q, s)[others])
    print(f"{name:17s}: scale {float(s):.4f} | distinct codes used {len(np.unique(q)):3d} | "
          f"values rounded to 0: {np.mean(q[others] == 0):5.1%} | error on the ordinary values {err:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2), sharey=True)
for ax, (name, v) in zip(axes, [("no outlier", vals), ("one outlier", with_outlier)]):
    q, _ = quant_absmax(v)
    ax.hist(q, bins=np.arange(-128, 129, 2)); ax.set_title(f"int8 codes used - {name}")
    ax.set_xlabel("int8 code")
axes[0].set_ylabel("count"); plt.tight_layout(); plt.show()

With the outlier, the ordinary values use only a few codes around 0. In LLMs this is not hypothetical: beyond ~6.7B parameters, a few *feature dimensions* of the activations become systematically huge (magnitude 20-100) in almost every token. **LLM.int8()** (Dettmers et al., the method behind `load_in_8bit`) splits the matrix multiply: the few outlier columns of X (and matching rows of W) run in fp16, everything else in int8 with vector-wise scales (per row of X, per column of W).

In [ ]:
T_tok, D_in, D_out = 64, 512, 256
X = rng.normal(size=(T_tok, D_in)).astype(np.float32)
outlier_cols = [17, 300]
X[:, outlier_cols] *= 40.0                                                  # systematic outlier features
Wm = (rng.normal(size=(D_in, D_out)) / np.sqrt(D_in)).astype(np.float32)
Y = X @ Wm

def int8_matmul_vectorwise(Xa, Wa):
    qx, sx = quant_absmax(Xa, 8, axis=1)                                    # a scale per row of X (per token)
    qw, sw = quant_absmax(Wa.T, 8, axis=1)                                  # a scale per column of W
    acc = qx.astype(np.int32) @ qw.astype(np.int32).T                       # integer matmul, int32 accumulate
    return acc * sx * sw.T

qx, sx = quant_absmax(X); qw, sw = quant_absmax(Wm)
naive = (qx.astype(np.int32) @ qw.astype(np.int32)) * sx * sw               # one scale per tensor
vectorwise = int8_matmul_vectorwise(X, Wm)
is_out = np.abs(X).max(axis=0) > 6.0                                        # LLM.int8 threshold
mixed = X[:, is_out] @ Wm[is_out] + int8_matmul_vectorwise(X[:, ~is_out], Wm[~is_out])
res = {"int8 per-tensor": rel_err(Y, naive), "int8 vector-wise": rel_err(Y, vectorwise),
       f"LLM.int8 mixed ({is_out.sum()} fp16 cols)": rel_err(Y, mixed)}
for k, v in res.items():
    print(f"{k:28s} relative error of X @ W: {v:.4f}")
errs = list(res.values())
assert errs[2] < errs[1] < errs[0], "decomposing out the outlier features wins"

Keeping 2 of 512 columns in fp16 (0.4% of the work) cuts the error several-fold. Later methods attack the same problem differently: **SmoothQuant** rescales channels to move the outliers from activations into the weights; **AWQ** protects the weight channels that see large activations.

## 6. 4-bit: uniform int4 vs NF4

At 4 bits there are only 16 levels, so *where* they sit matters. Trained weights are roughly normally distributed: most values near 0, few in the tails. Uniform int4 spaces levels evenly; **NF4 (NormalFloat-4, from QLoRA)** places them at quantiles of a normal distribution so each level is used about equally often. Construction (as in bitsandbytes): 8 positive levels from normal quantiles, 7 negative, plus an exact zero, then scale to [-1, 1].

In [ ]:
def nf4_levels(offset=0.9677083):
    pos = norm.ppf(np.linspace(offset, 0.5, 9)[:-1])        # 8 positive quantiles
    neg = -norm.ppf(np.linspace(offset, 0.5, 8)[:-1])       # 7 negative quantiles
    lv = np.sort(np.concatenate([pos, [0.0], neg]))
    return lv / lv.max()

NF4 = nf4_levels()
INT4 = np.arange(-7, 8) / 7.0                               # symmetric int4: 15 evenly spaced levels
print("NF4 levels:", NF4.round(4))
assert len(NF4) == 16 and 0.0 in NF4 and NF4.min() == -1 and NF4.max() == 1

try:
    import bitsandbytes.functional as bnbF
    bnb_nf4 = bnbF.get_4bit_type("nf4", device="cpu").numpy()
    assert np.allclose(NF4, bnb_nf4, atol=1e-6)
    print("matches bitsandbytes.functional.get_4bit_type('nf4') exactly")
except ImportError as e:
    bnbF = None
    print(f"bitsandbytes not available ({e}); skipping the library check")

In [ ]:
def quant_codebook(w, levels, block=64):
    """Block-wise absmax, then round each value to the NEAREST level of the codebook."""
    flat = w.reshape(-1, block)
    absmax = np.abs(flat).max(axis=1, keepdims=True)
    idx = np.abs((flat / absmax)[..., None] - levels).argmin(-1)            # 4-bit index per value
    return (levels[idx] * absmax).reshape(w.shape), idx

W4 = rng.normal(0, 0.02, size=(512, 512)).astype(np.float32)                 # "trained" weights ~ normal
w_int4, idx_int4 = quant_codebook(W4, INT4)
w_nf4, idx_nf4 = quant_codebook(W4, NF4)
mse = {"int4 (uniform)": float(np.mean((W4 - w_int4) ** 2)), "NF4": float(np.mean((W4 - w_nf4) ** 2))}
print({k: f"{v:.3e}" for k, v in mse.items()}, f"-> NF4 error is {mse['NF4'] / mse['int4 (uniform)']:.0%} of int4's")
assert mse["NF4"] < mse["int4 (uniform)"]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
grid = np.linspace(-1, 1, 400)
axes[0].plot(grid, norm.pdf(grid * 2.5), color="gray", label="weight density (block-normalised)")
axes[0].vlines(INT4, 0, 0.1, color="tab:blue", label="int4 levels")
axes[0].vlines(NF4, 0.1, 0.2, color="tab:red", label="NF4 levels")
axes[0].set_title("NF4 packs levels where the weights are"); axes[0].legend(fontsize=8)
axes[1].bar(np.arange(15) - 0.2, np.bincount(idx_int4.ravel(), minlength=15) / idx_int4.size, 0.4, label="int4")
axes[1].bar(np.arange(16) + 0.2, np.bincount(idx_nf4.ravel(), minlength=16) / idx_nf4.size, 0.4, label="NF4")
axes[1].set_xlabel("level index"); axes[1].set_ylabel("share of weights"); axes[1].legend()
axes[1].set_title("How often each 4-bit code is used")
plt.tight_layout(); plt.show()

Uniform int4 wastes its outer levels (few weights live there) and crowds the middle; NF4's levels are used much more evenly, so the same 4 bits carry more information. Now the library check: bitsandbytes' real `quantize_4bit` must produce exactly our numbers.

In [ ]:
if bnbF is not None:
    wt = torch.tensor(W4)
    packed, state = bnbF.quantize_4bit(wt, blocksize=64, quant_type="nf4")
    w_bnb = bnbF.dequantize_4bit(packed, state).numpy()
    print(f"bitsandbytes stores {packed.numel():,} bytes for {wt.numel():,} weights (2 per byte) + "
          f"{state.absmax.numel():,} fp32 absmax values")
    assert np.allclose(w_bnb, w_nf4, atol=1e-6), "our NF4 == bitsandbytes NF4"
    print(f"our from-scratch NF4 == bitsandbytes dequantize_4bit(quantize_4bit(W)): max diff {np.abs(w_bnb - w_nf4).max():.1e}")

### Double quantization: the memory math

Block-wise 4-bit needs one absmax constant per block. In fp32 with blocks of 64 that is 32/64 = **0.5 extra bits per weight** - 12% on top of 4 bits. QLoRA's *double quantization* quantizes those constants too (8-bit, in blocks of 256, with one fp32 constant per block of constants):

In [ ]:
plain = 32 / 64
double = 8 / 64 + 32 / (64 * 256)
saved_gb_65b = (plain - double) * 65e9 / 8 / 1e9
print(f"scale overhead: {plain:.3f} bits/weight -> {double:.3f} bits/weight with double quantization")
print(f"saving on a 65B model: {saved_gb_65b:.2f} GB")
for n in [7e9, 70e9]:
    print(f"{n / 1e9:.0f}B in NF4 + double quant: {(4 + double) * n / 8 / 1e9:.1f} GB (vs {n * 2 / 1e9:.0f} GB in bf16)")
assert abs(double - 0.127) < 1e-3 and abs(saved_gb_65b - 3.0) < 0.1

That is QLoRA's recipe: the frozen base model in NF4 + double quantization (a 65B model in ~33 GB), computation in bf16 (weights are dequantized on the fly per layer), and small LoRA adapters trained in 16-bit on top. See [QLoRA with bitsandbytes and PEFT](../11_finetuning/05_qlora_with_bitsandbytes_and_peft.ipynb).

## 7. Weight-only vs weight + activation quantization

| Scheme | Weights | Activations / matmul | Speeds up | Typical use |
|---|---|---|---|---|
| **W8A16 / W4A16** (weight-only) | int8 / 4-bit | dequantized to fp16, fp16 matmul | memory-bound decode (fewer bytes to read) | GPTQ, AWQ, bitsandbytes, GGUF - small-batch serving, laptops |
| **W8A8** (int8 or FP8) | int8 / FP8 | int8 / FP8 matmul | compute-bound prefill and large batches (2x tensor-core throughput) | SmoothQuant, FP8 on H100 in vLLM/TensorRT-LLM |
| **KV-cache quantization** (FP8/INT8) | - | the cached K and V | long context / big batch memory | vLLM `kv_cache_dtype="fp8"` - separate from weight quantization (tr33) |

Activations are harder to quantize than weights (outliers, and they change with every input), which is why 4-bit is common for weights but activations rarely go below 8 bits.

## 8. Accuracy vs bits on a real (small) trained model

Train a small MLP on the scikit-learn digits dataset, then quantize its weights at 8, 6, 4, 3 and 2 bits (quantize -> dequantize, "fake quantization") and re-measure test accuracy.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

digits = load_digits()
Xtr, Xte, ytr, yte = train_test_split(digits.data / 16.0, digits.target, test_size=0.3, random_state=0,
                                      stratify=digits.target)
Xtr_t, Xte_t = torch.tensor(Xtr, dtype=torch.float32), torch.tensor(Xte, dtype=torch.float32)
ytr_t, yte_t = torch.tensor(ytr), torch.tensor(yte)

torch.manual_seed(0)
mlp = nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 10))
opt = torch.optim.Adam(mlp.parameters(), lr=3e-3)
for epoch in range(150):                                                    # full-batch, ~2 s on CPU
    loss = nn.functional.cross_entropy(mlp(Xtr_t), ytr_t)
    opt.zero_grad(); loss.backward(); opt.step()

def accuracy(model):
    with torch.no_grad():
        return float((model(Xte_t).argmax(1) == yte_t).float().mean())

fp32_acc = accuracy(mlp)
print(f"fp32 test accuracy: {fp32_acc:.3f} ({sum(p.numel() for p in mlp.parameters()):,} parameters)")

In [ ]:
import copy

def quantized_copy(model, bits, per_channel=True):
    qm = copy.deepcopy(model)
    with torch.no_grad():
        for layer in qm:
            if isinstance(layer, nn.Linear):
                wq = layer.weight.numpy()
                q, s = quant_absmax(wq, bits, axis=1 if per_channel else None)   # a scale per output channel
                layer.weight.copy_(torch.tensor(dequant(q, s)))
    return qm

bit_list = [8, 6, 4, 3, 2]
acc_pc = [accuracy(quantized_copy(mlp, b, per_channel=True)) for b in bit_list]
acc_pt = [accuracy(quantized_copy(mlp, b, per_channel=False)) for b in bit_list]
print(pd.DataFrame({"bits": bit_list, "per-channel acc": acc_pc, "per-tensor acc": acc_pt,
                    "weight KB": [sum(p.numel() for p in mlp.parameters()) * b / 8 / 1024 for b in bit_list]})
      .round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(bit_list, acc_pc, "o-", label="per-channel scales")
ax.plot(bit_list, acc_pt, "s--", label="per-tensor scale")
ax.axhline(fp32_acc, color="gray", ls=":", label=f"fp32 ({fp32_acc:.3f})")
ax.set_xlabel("bits per weight"); ax.set_ylabel("test accuracy"); ax.invert_xaxis()
ax.set_title("Accuracy vs bits: flat down to ~4 bits, then a cliff"); ax.legend(); ax.grid(alpha=0.3)
plt.show()
assert abs(acc_pc[0] - fp32_acc) <= 0.01, "8-bit is essentially lossless"
assert acc_pc[-1] < acc_pc[0], "2 bits costs accuracy"

The shape of this curve is the general lesson: **8-bit is essentially free, 4-bit costs little (with good scales), 3-bit starts to hurt and 2-bit falls off a cliff** - especially with a single per-tensor scale - unless you use smarter methods (GPTQ's error compensation, AWQ's channel protection, mixed precision). Big LLMs tolerate 4-bit better than tiny models, because each weight matters less.

## 9. The practical landscape

| Method | What it is | Needs calibration data? | Bits | Runs on | Pick it when |
|---|---|---|---|---|---|
| **bitsandbytes** (LLM.int8, NF4/FP4) | on-the-fly quantization at load time (`BitsAndBytesConfig`) | no | 8, 4 | GPU (CPU backend exists) | QLoRA fine-tuning; quick experiments; no time to prepare a quantized checkpoint |
| **GPTQ** | layer-by-layer quantization that compensates each rounding error using second-order (Hessian) info | yes (~128 samples) | 4, 3, 8 | GPU (vLLM, TGI, ExLlama) | fast GPU serving of 4-bit models |
| **AWQ** | activation-aware: scales up the ~1% of weight channels that see large activations before quantizing | yes (small) | 4 | GPU (vLLM, TGI) | 4-bit GPU serving; often slightly better quality than GPTQ |
| **GGUF k-quants** (llama.cpp, e.g. `Q4_K_M`, `Q5_K_M`, `Q8_0`) | block-wise quantization with super-blocks; `_M` keeps sensitive tensors at higher precision | no (optional importance matrix) | 2-8 | CPU, Apple Silicon, consumer GPUs (llama.cpp, Ollama, LM Studio) | laptops and local inference; `Q4_K_M` is the usual sweet spot |
| **FP8** (E4M3) | 8-bit floating point weights + activations | minimal | 8 | H100/L40S/MI300 | high-throughput serving with ~no quality loss |

Rules of thumb: 8-bit ≈ lossless; 4-bit loses ~1-3% on hard benchmarks for 7B+ models (more for small models, more for maths/code); always evaluate on *your* task. Real bitsandbytes loading is in [Quantization with bitsandbytes](../10_huggingface/06_quantization_with_bitsandbytes.ipynb).

```python
# What it looks like with Hugging Face (see 10_huggingface/06 for a runnable version):
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
model = AutoModelForCausalLM.from_pretrained("some/model", quantization_config=bnb, device_map="auto")
```

## Try it yourself

**1.** Block size trade-off: for 4-bit absmax quantization of `W` (section 4), sweep block sizes 16, 32, 64, 128, 256. Print the error and the scale overhead in bits/weight (fp16 scales). Where is the sweet spot?

In [ ]:
# Solution — try it yourself first, then run this
for blk in [16, 32, 64, 128, 256]:
    e = rel_err(W, quant_blockwise(W, 4, block=blk))
    print(f"block {blk:3d}: relative error {e:.4f} | overhead {16 / blk:.3f} bits/weight -> {4 + 16 / blk:.3f} bits total")

**2.** (in30) Serve a 70B model (80 layers, 8 KV heads, head_dim 128) on one 80 GB GPU. For fp16, int8 and 4-bit weights, how many concurrent 8k-token sequences fit (fp16 KV cache, 10% overhead)?

In [ ]:
# Solution — try it yourself first, then run this
kv_per_seq = 2 * 80 * 8 * 128 * 8192 * 2                        # bytes, fp16
for name, bpp in [("fp16", 2), ("int8", 1), ("4-bit", 0.5 + 0.127 / 8)]:
    free = 80e9 * 0.9 - 70e9 * bpp
    n = int(free // kv_per_seq) if free > 0 else 0
    print(f"{name:5s}: weights {70e9 * bpp / 1e9:5.1f} GB -> {max(free, 0) / 1e9:5.1f} GB for cache -> {n} sequences of 8k")

**3.** Replace uniform int4 with NF4 in the MLP (block-wise, block 64) and compare test accuracy at 4 bits with per-channel int4.

In [ ]:
# Solution — try it yourself first, then run this
nf4_mlp = copy.deepcopy(mlp)
with torch.no_grad():
    for layer in nf4_mlp:
        if isinstance(layer, nn.Linear):
            wq = layer.weight.numpy()
            layer.weight.copy_(torch.tensor(quant_codebook(wq.reshape(-1, 64), NF4, block=64)[0].reshape(wq.shape)))
print(f"4-bit accuracy: per-channel int4 {acc_pc[bit_list.index(4)]:.3f} | block-wise NF4 {accuracy(nf4_mlp):.3f} | fp32 {fp32_acc:.3f}")

## Say it in an interview

- **One paragraph (fo30):** "Quantization stores weights (and sometimes activations or the KV cache) in fewer bits - 8 or 4 instead of 16 - using a scale per tensor, channel or block to map floats onto a small integer grid. It cuts memory 2-4x and speeds up memory-bound decoding; the cost is rounding error, negligible at 8 bits, small at 4 bits with good methods, severe below."
- **Memory maths:** params x bytes (+ KV cache + ~10-20% overhead). 7B: 28 GB fp32, 14 GB fp16, 7 GB int8, ~3.5-4 GB 4-bit. 70B fp16 = 140 GB -> needs 2 GPUs or 4-bit.
- **Mechanics:** absmax scale = max|w|/127; zero-point for skewed data; finer granularity (per-channel, block of 64) beats per-tensor; outliers ruin shared scales (LLM.int8 keeps outlier features in fp16).
- **NF4:** 16 levels at normal quantiles, block-wise absmax, exact zero; lower error than uniform int4 on normal weights; double quantization cuts scale overhead from 0.5 to 0.127 bits/weight (~3 GB on 65B).
- **Choosing:** GPU serving -> AWQ/GPTQ 4-bit or FP8; laptop/CPU -> GGUF `Q4_K_M`; fine-tuning on one GPU -> QLoRA (bitsandbytes NF4). Always re-evaluate on your task.
- **Traps:** weight quantization does not shrink the KV cache (tr33); "4-bit" files are ~4.5 bits/weight after scales; quantized weights are dequantized for the matmul in weight-only schemes, so they save memory and bandwidth, not FLOPs; small models degrade more than large ones.

## Next

- [Quantization with bitsandbytes](../10_huggingface/06_quantization_with_bitsandbytes.ipynb) - load a real model in 8 and 4 bits.
- [QLoRA with bitsandbytes and PEFT](../11_finetuning/05_qlora_with_bitsandbytes_and_peft.ipynb) - fine-tune on top of an NF4 base.
- [06 · KV cache and inference speed](06_kv_cache_and_inference_speed.ipynb) - the other half of serving memory.
- Theory: [genai_flow course](../../genai_flow/index.html) (chapter 18b: why the 3B model is 12.9 GB) · [interview bank](../../ai_interview_prep/index.html)